# NutriScan: AI-Powered Food Detection & Classification
### Capstone Project (CV) | Milestone 1: Data Understanding, EDA, Food Detection and Dataset Preparation

# Problem Statement

## Business Context

In today's fast-paced world, maintaining a balanced diet is a challenge for many people. Individuals struggle to track what they eat accurately, which leads to inconsistent nutritional habits and difficulty meeting health goals. Manual meal logging is time-consuming, error-prone and offers no real-time insight.

**NutriScan** is a smart food recognition system that uses image analysis and machine learning to identify food items from a photo and return real-time nutritional information (calories, macronutrients, portion size). It also suggests healthier alternatives based on user preferences and goals.

## Objective

As the Data Scientist at NutriScan, build an AI-powered food detection and classification system that:

1. **Detects and locates** food items in user-captured images (object detection).
2. **Classifies** each detected item into a food type (image classification).

These two capabilities unlock the downstream business outcomes (not built in this milestone):

- Personalised dietary recommendations and healthier meal alternatives from patterns in user food choices.
- Integration with health-tracking apps so users can monitor calorie intake and nutritional balance over time.
- Effortless tracking: one photo replaces manual logging.

## Why this matters (rationale for the report)

| Pain point today | What NutriScan changes | Value to the company |
|---|---|---|
| Manual logging is slow and users quit | One photo, instant result | Higher engagement and retention |
| Self-reported intake is inaccurate | Image-based identification | More trustworthy data, better recommendations |
| Generic diet advice | Personalised suggestions from logged habits | Differentiation, premium-tier potential |
| Siloed health data | Integration with health apps | Partnerships and ecosystem reach |

**Proposed success targets (to be agreed with stakeholders):** usable food crop for at least 90% of images; classification top-1 accuracy of at least 85% (Milestone 2).

## Milestone 1 Scope and Rubric Map (20 points)

| Rubric item | Pts | Notebook section | Report section |
|---|---|---|---|
| Introduction of the business problem | 2 | Problem Statement (this section) | 1. Business problem and rationale |
| Exploratory Data Analysis | 4 | Section 3 | 2. Data overview, 3. Key EDA findings |
| Object Detection (detect, crop, preprocess, build dataset) | 10 | Sections 4 and 5 | 4. Detection and preprocessing approach |
| Actionable Insights | 4 | Section 6 | 5. Insights and risks |
| Plan for the classification phase | - | Section 7 | 6. Methodology, 7. Next steps |

## Analytical Problem Framing

| Business question | Analytical task | Method | Output / metric |
|---|---|---|---|
| Where is the food in the photo? | Object detection | Zero-shot detector (OWL-ViT) with text prompts, because Food-101 has class labels but **no bounding boxes** | Boxes, detection rate, confidence |
| What food is it? | Image classification | Transfer learning (Milestone 2) | Top-1/Top-5 accuracy, macro-F1 |
| Is the data fit for training? | EDA and data quality | Class balance, sizes, brightness, blur, duplicates | Insights and cleaning decisions |

## Dataset

**Food-101**: 101 food categories organised as class folders of JPEG images (the public release has 1,000 images per class, with training images deliberately left slightly noisy). Your copy is 772 MB, so it is probably a reduced or resized version. **This notebook discovers the actual structure instead of assuming it.**

## Project Design (pipeline)

```
Photo -> [Detector: finds food region] -> crop (+5% padding, squared)
      -> resize 224x224 -> save as classification dataset (train/val/test)
      -> [Classifier, Milestone 2] -> food label -> nutrition lookup -> recommendations
```

# Instructions

## Please read the instructions carefully before starting

- Run cells **sequentially from the top**. Use a **GPU runtime** (Runtime > Change runtime type > T4 GPU).
- Edit settings **only in the Configuration cell**. The most important are `DRIVE_ZIP_PATH` and `MAX_PER_CLASS`.
- Because the dataset is large, do a **smoke test first** with `MAX_PER_CLASS = 10`, then raise it (e.g. 150, or `None` for everything).
- Detections are checkpointed to Google Drive, so if Colab disconnects you can re-run and it resumes.
- Every figure is saved to `figures/` and key numbers are stored in the `R` dictionary, then written to `milestone1_results.json`. Use these when writing the PDF report.
- After each major output, write your observations in the markdown cells marked **Observations**.
- Final submission: HTML export of this notebook (see the last cell) and a PDF business report.

# 1. Installing and Importing Libraries

In [ ]:
# Single install cell. transformers is pinned for reproducible detector behaviour.
# Colab already has torch, torchvision, pandas, scikit-learn, seaborn, scipy.
%pip install -q "transformers==4.46.3"

**Note:** If you later see a numpy/binary-compatibility error, restart the runtime once and run from the next cell.

In [ ]:
import os, sys, json, math, time, random, shutil, hashlib, zipfile, subprocess, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from scipy import ndimage
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import nms
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")      # keep the submitted notebook clean
sns.set_style("whitegrid")

## Configuration
All tunable settings live here. Each has a short comment describing what to change.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Paths ------------------------------------------------------------------
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Food_101.zip"   # <-- where you uploaded the zip
EXTRACT_DIR = Path("/content/food101_raw")   Ca            # fast local disk (not Drive)
WORK_DIR    = Path("/content/nutriscan")
CROP_DIR    = WORK_DIR / "cropped"                       # cropped + resized images
DATASET_DIR = WORK_DIR / "classification_dataset"        # train/val/test folders
FIG_DIR     = WORK_DIR / "figures"                       # figures for the report
PERSIST_DIR = Path("/content/drive/MyDrive/NutriScan_M1") if IN_COLAB else WORK_DIR / "persist"

# ---- Data volume controls (dataset is large) --------------------------------
MAX_PER_CLASS    = 150     # images per class to process. 10 = smoke test, None = all
EDA_STATS_SAMPLE = 3000    # images used for brightness/blur statistics
SWEEP_PER_CLASS  = 3       # images per class for the detection-threshold sweep

# ---- Detector ---------------------------------------------------------------
DETECTOR_NAME  = "google/owlvit-base-patch32"   # accuracy upgrade: "google/owlv2-base-patch16-ensemble"
FOOD_PROMPTS   = ["a photo of food", "a plate of food", "a bowl of food", "a dish"]
DET_MIN_SCORE  = 0.03     # lowest score kept when running the detector (sweep happens afterwards)
DET_THRESHOLD  = 0.10     # final threshold, auto-suggested after the sweep
TARGET_DET_RATE = 0.90    # business target: share of images with a usable detection
NMS_IOU        = 0.5      # overlap above which boxes are merged
TOPK_BOXES     = 5        # boxes stored per image
DET_BATCH      = 16       # lower to 8 if the GPU runs out of memory
MIN_AREA_FRAC  = 0.10     # boxes smaller than 10% of the image are treated as noise

# ---- Crop / preprocessing ---------------------------------------------------
IMG_SIZE = 224            # EfficientNet/ResNet input size
CROP_PAD = 0.05           # 5% padding around each box
JPEG_QUALITY = 95
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---- Split ------------------------------------------------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

for d in (WORK_DIR, CROP_DIR, DATASET_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Helper functions
- **load_rgb:** opens an image, fixes phone-rotation (EXIF), converts to RGB, optionally makes a fast thumbnail.
- **show_grid / save_fig:** consistent plotting and saving every figure for the PDF report.
- **R / rset / save_results:** a results registry holding the key numbers for the report.

In [ ]:
R = {}   # RESULTS registry: key numbers used later in the PDF report

def rset(key, value):
    """Store a value in R, converting numpy types so it can be saved as JSON."""
    if isinstance(value, np.integer): value = int(value)
    elif isinstance(value, np.floating): value = float(value)
    R[key] = value

def save_results():
    with open(PERSIST_DIR / "milestone1_results.json", "w") as f:
        json.dump(R, f, indent=2, default=str)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

def load_rgb(path, max_side=None):
    """Open image as RGB with EXIF orientation fixed; optional fast thumbnail."""
    im = Image.open(path)
    if max_side:
        im.draft("RGB", (max_side, max_side))     # fast JPEG downscale on decode
    im = ImageOps.exif_transpose(im).convert("RGB")
    if max_side:
        im.thumbnail((max_side, max_side))
    return im

def show_grid(paths, titles=None, ncols=5, cell=3.0, title=None, fontsize=8, fname=None, max_side=300):
    n = len(paths); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * cell, nrows * cell))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for i, p in enumerate(paths):
        try: axes[i].imshow(load_rgb(p, max_side))
        except Exception: pass
        if titles is not None: axes[i].set_title(str(titles[i]), fontsize=fontsize)
    if title: fig.suptitle(title, fontsize=13)
    if fname: save_fig(fname)
    else: plt.tight_layout(); plt.show()

# 2. Loading the Data

## Mount Google Drive and extract the dataset
- **Mount:** gives Colab access to the zip stored in Drive (also used to save outputs that survive disconnects).
- **Extract to local disk:** `/content` is much faster than reading images from Drive. The extraction is skipped if already done.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

def extract_zip(zip_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    if next(out_dir.rglob("*.jp*g"), None) is not None:
        print("Already extracted:", out_dir); return
    assert Path(zip_path).exists(), f"Zip not found: {zip_path}. Fix DRIVE_ZIP_PATH."
    t0 = time.time()
    if shutil.which("unzip"):
        subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(out_dir)], check=True)
    else:
        with zipfile.ZipFile(zip_path) as z: z.extractall(out_dir)
    print(f"Extracted in {time.time() - t0:.0f}s")

extract_zip(DRIVE_ZIP_PATH, EXTRACT_DIR)

## Discover the folder structure
We do not assume the layout. The code lists the top-level folders, finds every image, and infers the **class label from the folder name** (handles `class/img.jpg`, `split/class/img.jpg` and `class/split/img.jpg`).

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES = {"train", "test", "val", "valid", "validation"}

# Quick look at the top of the folder tree
for p in sorted(EXTRACT_DIR.iterdir())[:10]:
    print("-", p.name, "(dir)" if p.is_dir() else "")
    if p.is_dir():
        for q in sorted(p.iterdir())[:5]: print("     ", q.name)

all_files = [p for p in EXTRACT_DIR.rglob("*")
             if p.is_file() and p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts]
print("\nImage files found:", len(all_files))
assert all_files, "No images found. Check the zip contents / EXTRACT_DIR."

parent_names = {p.parent.name.lower() for p in all_files}
if parent_names <= SPLIT_NAMES:             # layout: class/split/img
    label_of = lambda p: p.parent.parent.name
    split_of = lambda p: p.parent.name.lower()
else:                                       # layout: class/img  or  split/class/img
    label_of = lambda p: p.parent.name
    split_of = lambda p: p.parent.parent.name.lower() if p.parent.parent.name.lower() in SPLIT_NAMES else None

df = pd.DataFrame({"path": [str(p) for p in all_files],
                   "label": [label_of(p) for p in all_files],
                   "orig_split": [split_of(p) for p in all_files]})
print("Classes found:", df.label.nunique())
display(df.sample(5, random_state=SEED))

## Image metadata scan
For every image we record size, format, file size, an **MD5 hash** (exact-duplicate check) and whether it is **corrupt**. It uses threads and is cached to Drive.

In [ ]:
META_CSV = PERSIST_DIR / "metadata.csv"

def scan_image(p):
    try:
        with Image.open(p) as im:
            w, h, fmt, mode = im.width, im.height, im.format, im.mode
            im.verify()                                    # raises if the file is damaged
        with open(p, "rb") as f: md5 = hashlib.md5(f.read()).hexdigest()
        return dict(path=p, width=w, height=h, fmt=fmt, mode=mode,
                    file_kb=os.path.getsize(p) / 1024, md5=md5, corrupt=False)
    except Exception:
        return dict(path=p, width=np.nan, height=np.nan, fmt=None, mode=None,
                    file_kb=np.nan, md5=None, corrupt=True)

if META_CSV.exists() and len(pd.read_csv(META_CSV)) == len(df):
    meta = pd.read_csv(META_CSV); print("Loaded cached metadata")
else:
    with ThreadPoolExecutor(16) as ex:
        meta = pd.DataFrame(list(tqdm(ex.map(scan_image, df.path), total=len(df))))
    meta.to_csv(META_CSV, index=False)

df = df.merge(meta, on="path")
df["aspect"] = df.width / df.height
df["max_side"] = df[["width", "height"]].max(axis=1)
df["megapixels"] = df.width * df.height / 1e6
display(df.head())

## Data overview and quality checks
- **Overview:** images, classes, sizes, formats.
- **Corrupt files:** removed.
- **Exact duplicates:** removed (keeping the first), because duplicates across train/test would leak and inflate accuracy.
- **Working set:** capped at `MAX_PER_CLASS` per class so processing time stays manageable.

In [ ]:
df_ok = df[~df.corrupt].copy()
dup_mask = df_ok.duplicated("md5", keep="first")
conflict = (df_ok[df_ok.md5.duplicated(keep=False)].groupby("md5").label.nunique() > 1).sum()
df_clean = df_ok[~dup_mask]

shuffled = df_clean.sample(frac=1, random_state=SEED)
df_work = (shuffled.groupby("label").head(MAX_PER_CLASS) if MAX_PER_CLASS else shuffled).reset_index(drop=True)

overview = pd.DataFrame({"Metric": [
    "Total image files", "Classes", "Corrupt / unreadable", "Exact duplicates removed",
    "Duplicate groups with conflicting labels", "Images after cleaning", "Working set used (cap per class)",
    "Total size (MB)", "Median width x height", "Formats"],
 "Value": [len(df), df.label.nunique(), int(df.corrupt.sum()), int(dup_mask.sum()), int(conflict),
    len(df_clean), f"{len(df_work)} ({MAX_PER_CLASS})", round(df.file_kb.sum() / 1024, 1),
    f"{int(df.width.median())} x {int(df.height.median())}", ", ".join(map(str, df.fmt.dropna().unique()))]})
display(overview)

for k, v in [("n_images_total", len(df)), ("n_classes", df.label.nunique()), ("n_corrupt", df.corrupt.sum()),
             ("n_duplicates", dup_mask.sum()), ("n_clean", len(df_clean)), ("n_working_set", len(df_work))]:
    rset(k, v)

# 3. Exploratory Data Analysis

## 3.1 Target (class) distribution
A bar chart of images per class for the full clean data and the working set. We also compute an **imbalance ratio** (largest class / smallest class).

In [ ]:
counts_full = df_clean.label.value_counts().sort_index()
counts_work = df_work.label.value_counts().sort_index()

fig, axes = plt.subplots(2, 1, figsize=(18, 8))
for ax, c, t in [(axes[0], counts_full, "Images per class (full, after cleaning)"),
                 (axes[1], counts_work, f"Images per class (working set, cap={MAX_PER_CLASS})")]:
    ax.bar(c.index, c.values, color="#2a7f62"); ax.set_title(t)
    ax.axhline(c.mean(), color="red", ls="--", lw=1, label=f"mean = {c.mean():.0f}")
    ax.tick_params(axis="x", rotation=90, labelsize=6); ax.legend()
save_fig("01_class_distribution")

ratio = counts_full.max() / counts_full.min()
print(f"Smallest class: {counts_full.idxmin()} ({counts_full.min()}) | largest: {counts_full.idxmax()} ({counts_full.max()})")
print(f"Imbalance ratio (full): {ratio:.2f}  | working-set ratio: {counts_work.max() / counts_work.min():.2f}")
rset("imbalance_ratio_full", ratio); rset("imbalance_ratio_work", counts_work.max() / counts_work.min())

**Observations (write after running):**
- Is the dataset balanced? Which classes are smallest or largest?
- Does the imbalance ratio justify class weights or oversampling?

## 3.2 Visual inspection of the images
- **Random classes, 5 images each:** shows within-class variety (angle, plating, background).
- **One image per class mosaic:** shows how visually similar some classes are (e.g. different pasta, cakes, meats).

In [ ]:
rng = np.random.default_rng(SEED)
classes = sorted(df_work.label.unique())
pick = rng.choice(classes, size=min(8, len(classes)), replace=False)
paths, titles = [], []
for c in pick:
    s = df_work[df_work.label == c].head(5)
    paths += s.path.tolist(); titles += [c] * len(s)
show_grid(paths, titles, ncols=5, cell=2.6, title="5 random images from 8 random classes", fname="02_sample_grid")

In [ ]:
first = df_work.groupby("label").head(1).sort_values("label")
show_grid(first.path.tolist(), first.label.tolist(), ncols=11, cell=1.7, fontsize=5, max_side=150,
          title="One image per class", fname="03_class_mosaic")

**Observations (write after running):**
- Is there usually one dish per image, or several? Plates, hands, tables, packaging, text?
- Which classes look alike (hard for a classifier)? Any mislabelled or noisy images?
- How varied are lighting, angle (top-down vs side) and background?

## 3.3 Image dimensions, aspect ratio and file size
Models need a fixed input size, so we check how much resizing will be required and whether aspect ratios vary.

In [ ]:
smp = df_ok.sample(min(5000, len(df_ok)), random_state=SEED)
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].scatter(smp.width, smp.height, s=4, alpha=.3); ax[0].set(title="Width vs height (px)", xlabel="width", ylabel="height")
ax[1].hist(smp.aspect, bins=40, color="#2a7f62"); ax[1].set(title="Aspect ratio (w/h)")
ax[2].hist(smp.max_side, bins=40, color="#2a7f62"); ax[2].set(title="Longest side (px)")
ax[3].hist(smp.file_kb, bins=40, color="#2a7f62"); ax[3].set(title="File size (KB)")
save_fig("04_image_sizes")

print(df_ok[["width", "height", "aspect", "file_kb"]].describe().round(2))
print("\nFormats:", df_ok.fmt.value_counts().to_dict(), "| Modes:", df_ok["mode"].value_counts().to_dict())
share_sq = ((df_ok.aspect > .95) & (df_ok.aspect < 1.05)).mean()
print(f"Share of near-square images: {share_sq:.1%}")
rset("median_width", df_ok.width.median()); rset("median_height", df_ok.height.median())

**Observations (write after running):**
- Are all images the same size? What is the common longest side (e.g. 512)?
- Mixed aspect ratios mean a plain resize would distort food, so we crop to a square instead (Section 5).
- Any non-RGB modes that need converting?

## 3.4 Brightness, contrast, colour and sharpness
Computed on a sample of `EDA_STATS_SAMPLE` working-set images (thumbnails for speed). We flag **dark** (brightness < 0.25), **overexposed** (> 0.80), **low contrast** (< 0.10) and **blurry** (bottom 5% sharpness) images. The dataset mean RGB is compared with the ImageNet mean used by pretrained models.

In [ ]:
def image_stats(p):
    im = load_rgb(p, 128)
    a = np.asarray(im, dtype=np.float32) / 255.0
    gray = a.mean(axis=2)
    sat = np.asarray(im.convert("HSV"), dtype=np.float32)[..., 1].mean() / 255.0
    return dict(path=p, r=a[..., 0].mean(), g=a[..., 1].mean(), b=a[..., 2].mean(),
                brightness=gray.mean(), contrast=gray.std(),
                sharpness=ndimage.laplace(gray * 255).var(), saturation=sat)

stat_paths = df_work.sample(min(EDA_STATS_SAMPLE, len(df_work)), random_state=SEED).path.tolist()
with ThreadPoolExecutor(8) as ex:
    df_stats = pd.DataFrame(list(tqdm(ex.map(image_stats, stat_paths), total=len(stat_paths))))
df_stats = df_stats.merge(df_work[["path", "label"]], on="path")

df_stats["dark"] = df_stats.brightness < 0.25
df_stats["bright"] = df_stats.brightness > 0.80
df_stats["low_contrast"] = df_stats.contrast < 0.10
df_stats["blurry"] = df_stats.sharpness <= df_stats.sharpness.quantile(0.05)

fig, ax = plt.subplots(1, 4, figsize=(20, 3.8))
for a_, col in zip(ax, ["brightness", "contrast", "sharpness", "saturation"]):
    a_.hist(df_stats[col], bins=40, color="#2a7f62"); a_.set_title(col.capitalize())
save_fig("05_brightness_contrast")

mean_rgb = df_stats[["r", "g", "b"]].mean().values
print("Dataset mean RGB:", mean_rgb.round(3), "| ImageNet mean:", IMAGENET_MEAN)
print(df_stats[["dark", "bright", "low_contrast", "blurry"]].mean().mul(100).round(1).rename("% of sampled images"))
for k in ["dark", "bright", "low_contrast", "blurry"]: rset(f"pct_{k}", df_stats[k].mean() * 100)

In [ ]:
ex_dark = df_stats.sort_values("brightness").head(10)
ex_blur = df_stats.sort_values("sharpness").head(10)
show_grid(ex_dark.path.tolist() + ex_blur.path.tolist(),
          [f"dark {v:.2f}" for v in ex_dark.brightness] + [f"blur {v:.0f}" for v in ex_blur.sharpness],
          ncols=5, cell=2.4, title="Darkest 10 (top) and blurriest 10 (bottom) sampled images", fname="05b_quality_examples")
cls_b = df_stats.groupby("label").brightness.mean().sort_values()
print("Darkest classes:", cls_b.head(5).round(2).to_dict())
print("Brightest classes:", cls_b.tail(5).round(2).to_dict())

**Observations (write after running):**
- What % of images are dark, overexposed, low-contrast or blurry? Are these likely to hurt detection or classification?
- How close is the dataset mean RGB to ImageNet's (supports using ImageNet normalisation)?
- Implication: augmentation (brightness/contrast jitter, blur) will make the model robust to real-world user photos.

# 4. Object Detection: Locating the Food

## Approach
Food-101 has **class labels only, no bounding boxes**, so a detector cannot be trained directly. A COCO-pretrained YOLO knows only a handful of food classes (pizza, cake, banana...) and would miss most dishes. We therefore use a **zero-shot, text-prompted detector (OWL-ViT)** queried with generic prompts such as "a plate of food". It needs no training and works for all 101 classes.

| Option | Pros | Cons | Decision |
|---|---|---|---|
| YOLOv8 (COCO) | Fast | Few food classes | Not used |
| **OWL-ViT zero-shot** | No boxes needed, covers all dishes | Slower, scores are low-valued | **Used** |
| Grounding DINO / YOLO-World | Similar zero-shot ability | Heavier / extra dependencies | Alternatives if accuracy is insufficient |
| Fine-tune YOLO on labelled boxes | Highest accuracy | Needs boxes (e.g. UEC-Food256) | Possible future step |

**Box selection rule:** keep the highest-scoring box that covers at least `MIN_AREA_FRAC` of the image. If none qualifies, **fall back to the full image** and log the reason. This guarantees every image yields a usable training sample.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Switch runtime to T4 GPU or use a very small MAX_PER_CLASS.")

processor = AutoProcessor.from_pretrained(DETECTOR_NAME)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_NAME).to(DEVICE).eval()
print("Detector ready:", DETECTOR_NAME)

## Detection functions
- **run_detector:** runs a batch of images, converts boxes to original pixel coordinates, applies NMS (merges overlapping boxes) and keeps the top-K.
- **detect_paths:** runs over many files with background image loading, stores results in the global `DET` dictionary and **checkpoints to Drive** so it can resume.
- **select_box:** applies the threshold and area rule, and records why a fallback happened.

In [ ]:
DET = {}   # path -> {"w", "h", "boxes", "scores"}
DET_CKPT = PERSIST_DIR / "detections.json"

@torch.no_grad()
def run_detector(images, min_score=DET_MIN_SCORE, topk=TOPK_BOXES):
    """images: list of PIL RGB. Returns per-image dict(boxes [N,4] xyxy in pixels, scores [N])."""
    inputs = processor(text=[FOOD_PROMPTS] * len(images), images=images, return_tensors="pt", padding=True).to(DEVICE)
    outputs = detector(**inputs)
    # OWLv2 pads images to a square, so its boxes are relative to the longest side
    sizes = [[max(im.size)] * 2 if "owlv2" in DETECTOR_NAME else [im.height, im.width] for im in images]
    res = processor.post_process_object_detection(outputs=outputs, threshold=min_score,
                                                  target_sizes=torch.tensor(sizes, device=DEVICE))
    out = []
    for r in res:
        b, s = r["boxes"].float().cpu(), r["scores"].float().cpu()
        if len(s):
            keep = nms(b, s, NMS_IOU)[:topk]
            b, s = b[keep], s[keep]
        out.append(dict(boxes=b.numpy(), scores=s.numpy()))
    return out

class PathDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return i, load_rgb(self.paths[i])

def _collate(batch): return batch

def detect_paths(paths, ckpt=None, batch_size=DET_BATCH):
    if ckpt is not None and ckpt.exists() and not DET:
        DET.update(json.load(open(ckpt))); print(f"Resumed {len(DET)} detections from checkpoint")
    todo = [p for p in paths if p not in DET]
    print(f"{len(todo)} images to process ({len(paths) - len(todo)} already done)")
    dl = DataLoader(PathDS(todo), batch_size=batch_size, num_workers=2, collate_fn=_collate)
    for bi, batch in enumerate(tqdm(dl)):
        idxs, imgs = zip(*batch)
        for i, im, o in zip(idxs, imgs, run_detector(list(imgs))):
            DET[todo[i]] = dict(w=im.width, h=im.height, boxes=o["boxes"].round(1).tolist(),
                                scores=o["scores"].round(4).tolist())
        if ckpt is not None and bi % 50 == 49:
            json.dump(DET, open(ckpt, "w"))
    if ckpt is not None: json.dump(DET, open(ckpt, "w"))

def select_box(rec, thr, min_area=MIN_AREA_FRAC):
    """Pick the best box. Returns dict(ok, reason, box, score, area_frac, n_regions)."""
    W, H = rec["w"], rec["h"]
    over = [(s, b) for b, s in zip(rec["boxes"], rec["scores"]) if s >= thr]
    cands = []
    for s, (x1, y1, x2, y2) in over:
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        a = max(0, x2 - x1) * max(0, y2 - y1) / (W * H)
        if a >= min_area: cands.append((s, a, [x1, y1, x2, y2]))
    if not over:   return dict(ok=False, reason="no_detection", box=None, score=0.0, area_frac=1.0, n_regions=0)
    if not cands:  return dict(ok=False, reason="box_too_small", box=None, score=0.0, area_frac=1.0, n_regions=0)
    s, a, b = max(cands, key=lambda t: t[0])
    return dict(ok=True, reason="", box=b, score=s, area_frac=a, n_regions=len(cands))

## 4.1 Visual check on sample images
Boxes above the threshold are drawn in yellow, and the **selected box in green**. This is the first sanity check that the detector finds the food.

In [ ]:
def show_detections(paths, ncols=5, cell=3.2, titles=None, fname=None, thr=None):
    thr = thr or DET_THRESHOLD
    n = len(paths); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * cell, nrows * cell))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for i, p in enumerate(paths):
        rec = DET[p]; sel = select_box(rec, thr)
        axes[i].imshow(load_rgb(p))
        for b, s in zip(rec["boxes"], rec["scores"]):
            if s >= thr:
                axes[i].add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1))
        if sel["ok"]:
            b = sel["box"]
            axes[i].add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=3))
            t = f"score {sel['score']:.2f} | area {sel['area_frac']:.0%}"
        else:
            t = f"FALLBACK: {sel['reason']}"
        axes[i].set_title((titles[i] + "\n" if titles else "") + t, fontsize=8)
    if fname: save_fig(fname)
    else: plt.tight_layout(); plt.show()

demo = df_work.sample(min(15, len(df_work)), random_state=1)
detect_paths(demo.path.tolist())
show_detections(demo.path.tolist(), titles=demo.label.tolist(), fname="06_detection_samples")

**Observations (write after running):** Do the green boxes cover the food (not just the plate or table)? Note good and bad cases for the report.

## 4.2 Choosing the detection threshold
We run the detector once on a stratified sample (low threshold), then evaluate several thresholds offline. A **higher threshold gives more precise boxes but more fallbacks**. We choose the highest threshold that still gives a usable detection for at least `TARGET_DET_RATE` of images.

In [ ]:
sweep_df = df_work.groupby("label").head(SWEEP_PER_CLASS)
detect_paths(sweep_df.path.tolist())

thresholds = [0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30]
rows = []
for t in thresholds:
    sels = [select_box(DET[p], t) for p in sweep_df.path]
    ok = [s for s in sels if s["ok"]]
    rows.append(dict(threshold=t, detection_rate=len(ok) / len(sels),
                     mean_score=np.mean([s["score"] for s in ok]) if ok else 0,
                     mean_area=np.mean([s["area_frac"] for s in ok]) if ok else 0))
sweep = pd.DataFrame(rows); display(sweep.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep.threshold, sweep.detection_rate, "o-", label="detection rate")
ax.plot(sweep.threshold, sweep.mean_area, "s--", label="mean box area fraction")
ax.axhline(TARGET_DET_RATE, color="red", ls=":", label=f"target {TARGET_DET_RATE:.0%}")
ax.set(xlabel="score threshold", title="Threshold sweep"); ax.legend()
save_fig("07_threshold_sweep")

good = sweep[sweep.detection_rate >= TARGET_DET_RATE]
DET_THRESHOLD = float(good.threshold.max()) if len(good) else float(sweep.threshold.min())
print(f"Selected DET_THRESHOLD = {DET_THRESHOLD} (override manually here if you prefer)")
rset("det_threshold", DET_THRESHOLD)

## 4.3 Run detection on the full working set
Runs on every image in the working set with checkpointing to Drive. If Colab disconnects, re-run this cell and it resumes.

In [ ]:
detect_paths(df_work.path.tolist(), ckpt=DET_CKPT)

## 4.4 Detection results and accuracy
- **Detection rate:** share of images where the detector found a usable food box.
- **Fallback rate:** share where we kept the full image (reason logged).
- **Multiple regions:** images with more than one candidate food box (multi-dish plates).

In [ ]:
rows = []
for p in df_work.path:
    s = select_box(DET[p], DET_THRESHOLD)
    rows.append(dict(path=p, ok=s["ok"], reason=s["reason"], score=s["score"], area_frac=s["area_frac"],
                     n_regions=s["n_regions"], box=s["box"], w=DET[p]["w"], h=DET[p]["h"]))
df_det = pd.DataFrame(rows).merge(df_work[["path", "label", "md5"]], on="path")

det_rate = df_det.ok.mean()
print(f"Detection rate: {det_rate:.1%} | fallback (full image) rate: {1 - det_rate:.1%}")
print(df_det[~df_det.ok].reason.value_counts().to_dict())
print(f"Mean score (detected): {df_det[df_det.ok].score.mean():.3f} | median box area: {df_det[df_det.ok].area_frac.median():.0%}")
print(f"Images with 2+ candidate food regions: {(df_det.n_regions >= 2).mean():.1%}")
rset("detection_rate", det_rate); rset("fallback_rate", 1 - det_rate)
rset("mean_det_score", df_det[df_det.ok].score.mean()); rset("median_box_area_frac", df_det[df_det.ok].area_frac.median())
rset("pct_multi_region", (df_det.n_regions >= 2).mean() * 100)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].hist(df_det[df_det.ok].score, bins=30, color="#2a7f62"); ax[0].set_title("Detection confidence")
ax[1].hist(df_det[df_det.ok].area_frac, bins=30, color="#2a7f62"); ax[1].set_title("Box area / image area")
worst = (1 - df_det.groupby("label").ok.mean()).sort_values(ascending=False).head(15)
ax[2].barh(worst.index[::-1], worst.values[::-1], color="#c0392b"); ax[2].set_title("15 classes with highest fallback rate")
save_fig("08_detection_summary")

**Observations (write after running):** What is the detection rate vs the 90% target? Which classes fall back most (soups, salads, sushi platters?) and why?

## 4.5 Manual accuracy check (no ground-truth boxes)
Without ground-truth boxes we cannot compute mAP, so we **manually review 50 random detections** and report a rough precision. Look at the grid, note the IDs of boxes that are **wrong** (`BAD`) or only **partly right** (`PARTIAL`), and list them below. All others count as good.

In [ ]:
qa = df_det[df_det.ok].sample(min(50, int(df_det.ok.sum())), random_state=7).reset_index(drop=True)
show_detections(qa.path.tolist(), ncols=5, cell=2.6, titles=[f"ID {i}" for i in range(len(qa))], fname="09_manual_qa_grid")

In [ ]:
# >>> Fill these after inspecting the grid above (IDs shown in each title) <<<
BAD_IDS = []        # e.g. [3, 17]  box misses the food completely
PARTIAL_IDS = []    # e.g. [8, 22]  box covers only part of the food or too much background

n = len(qa)
if not BAD_IDS and not PARTIAL_IDS:
    print("No review recorded yet. Fill BAD_IDS / PARTIAL_IDS above, then re-run this cell.")
else:
    good_n = n - len(BAD_IDS) - len(PARTIAL_IDS)
    print(f"Manual review of {n} detections: good {good_n/n:.0%} | partial {len(PARTIAL_IDS)/n:.0%} | bad {len(BAD_IDS)/n:.0%}")
    rset("manual_precision_good", good_n / n); rset("manual_precision_good_or_partial", (good_n + len(PARTIAL_IDS)) / n)

## 4.6 Where detection does not work well
We look at the fallbacks and the lowest-confidence accepted boxes, and check whether dark or blurry images are harder.

In [ ]:
fb = df_det[~df_det.ok].head(10)
lowc = df_det[df_det.ok].sort_values("score").head(10)
show_detections(fb.path.tolist() + lowc.path.tolist(), ncols=5, cell=2.6,
                titles=fb.label.tolist() + lowc.label.tolist(), fname="10_failure_cases")

chk = df_det.merge(df_stats[["path", "dark", "blurry", "low_contrast"]], on="path")
for flag in ["dark", "blurry", "low_contrast"]:
    g = chk.groupby(flag).ok.agg(["mean", "size"]).round(3)
    print(f"\nDetection rate by '{flag}':"); print(g)

**Observations (write after running):** What do the failures have in common (multiple dishes, soups with no edges, dark photos, close-ups where food fills the frame)? A fallback on a close-up is acceptable, since the full image is the food.

# 5. Cropping, Preprocessing and Building the Classification Dataset

## Crop and resize
For each image:
- **Crop:** the selected box plus 5% padding. The box is expanded to a **square** (when it fits inside the image) so resizing does not distort the food. Fallback images use the full image.
- **Resize:** to 224x224 (standard input size for EfficientNet/ResNet).
- **Save:** as JPEG in `cropped/<class>/`. Pixels are stored as uint8; **normalisation is applied on the fly** at training time (smaller files, no precision loss).

In [ ]:
def compute_crop_box(W, H, box, pad=CROP_PAD):
    """Padded, squared crop box inside the image. box=None means the full image."""
    if box is None: return (0, 0, W, H)
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    x1, y1, x2, y2 = x1 - pad * bw, y1 - pad * bh, x2 + pad * bw, y2 + pad * bh
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    bw, bh = x2 - x1, y2 - y1
    side = max(bw, bh)
    if side <= min(W, H):                                    # expand to a square, shift to stay inside
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1 = min(max(0, cx - side / 2), W - side); y1 = min(max(0, cy - side / 2), H - side)
        x2, y2 = x1 + side, y1 + side
    return (x1, y1, x2, y2)

def make_crop(row):
    try:
        im = load_rgb(row["path"])
        cb = compute_crop_box(im.width, im.height, row["box"])
        out = im.crop(tuple(int(round(v)) for v in cb)).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        dest = CROP_DIR / row["label"] / f"{Path(row['path']).stem}_{row['md5'][:6]}.jpg"
        out.save(dest, quality=JPEG_QUALITY)
        return str(dest)
    except Exception:
        return None

for c in df_det.label.unique(): (CROP_DIR / c).mkdir(parents=True, exist_ok=True)
recs = df_det.to_dict("records")
with ThreadPoolExecutor(8) as ex:
    df_det["crop_path"] = list(tqdm(ex.map(make_crop, recs), total=len(recs)))
df_det["used_fallback"] = ~df_det.ok

failed = df_det.crop_path.isna().sum()
df_crops = df_det.dropna(subset=["crop_path"]).reset_index(drop=True)
print(f"Crops created: {len(df_crops)} | failed: {failed} | fallback (full image): {df_crops.used_fallback.mean():.1%}")
rset("n_crops", len(df_crops))

## Before / after examples
Left: original with the chosen box. Right: the final 224x224 crop that goes to the classifier.

In [ ]:
ex = df_crops[df_crops.ok].sample(6, random_state=3) if df_crops.ok.sum() >= 6 else df_crops.head(6)
fig, axes = plt.subplots(2, 6, figsize=(18, 6.2))
for i, (_, r) in enumerate(ex.iterrows()):
    axes[0, i].imshow(load_rgb(r.path))
    if r.box is not None:
        b = r.box; axes[0, i].add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=3))
    axes[0, i].set_title(f"{r.label}\noriginal {r.w}x{r.h}", fontsize=8)
    axes[1, i].imshow(Image.open(r.crop_path)); axes[1, i].set_title(f"crop {IMG_SIZE}x{IMG_SIZE}", fontsize=8)
for a in axes.ravel(): a.axis("off")
save_fig("11_before_after")

## Normalisation
Pretrained models expect inputs scaled to [0,1] and standardised with the ImageNet mean/std. We verify the transform on a batch of crops and confirm it is reversible, and compare crop statistics with ImageNet's.

In [ ]:
norm_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

sample_crops = df_crops.sample(min(64, len(df_crops)), random_state=1).crop_path.tolist()
raw = torch.stack([transforms.ToTensor()(Image.open(p).convert("RGB")) for p in sample_crops])
normed = torch.stack([norm_tf(Image.open(p).convert("RGB")) for p in sample_crops])
print("Batch shape:", tuple(normed.shape))
print(f"Raw    -> min {raw.min():.2f} max {raw.max():.2f} mean {raw.mean():.3f}")
print(f"Normed -> min {normed.min():.2f} max {normed.max():.2f} mean {normed.mean():.3f}")
m, s = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
print("Reversible:", torch.allclose(normed * s + m, raw, atol=1e-5))
print("Crop mean RGB:", raw.mean((0, 2, 3)).numpy().round(3), "| std:", raw.std((0, 2, 3)).numpy().round(3))

## Build the train / validation / test dataset
- **Remove duplicates** (already done on the MD5 hash) so the same photo never appears in two splits.
- **Stratified split 70/15/15** keeps class proportions identical in every split.
- **Folder layout** `train|val|test/<class>/` is directly loadable by `ImageFolder` (PyTorch) or `image_dataset_from_directory` (Keras).

In [ ]:
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1) < 1e-9
train_df, temp_df = train_test_split(df_crops, test_size=1 - TRAIN_FRAC, stratify=df_crops.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=temp_df.label, random_state=SEED)
train_df, val_df, test_df = train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")
manifest = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

# Materialise folders (crops are small, so copying is fine)
shutil.rmtree(DATASET_DIR, ignore_errors=True)
def place(r):
    d = DATASET_DIR / r["split"] / r["label"]; d.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r["crop_path"], d / Path(r["crop_path"]).name)
with ThreadPoolExecutor(8) as ex:
    list(tqdm(ex.map(place, manifest.to_dict("records")), total=len(manifest)))

print(manifest.split.value_counts().to_dict())
assert set(train_df.md5).isdisjoint(val_df.md5) and set(train_df.md5).isdisjoint(test_df.md5), "Leakage found"
print("No duplicate images across splits.")
class_names = sorted(manifest.label.unique())
json.dump(class_names, open(PERSIST_DIR / "class_names.json", "w"))
manifest[["path", "crop_path", "label", "split", "ok", "score", "area_frac", "used_fallback"]].to_csv(PERSIST_DIR / "manifest.csv", index=False)
for k, v in manifest.split.value_counts().items(): rset(f"n_{k}", v)

## Verify the final dataset
Loads each split with `ImageFolder` + `DataLoader`, checks that class indices match across splits, and prints the final tensor shapes and per-split class counts.

In [ ]:
loaders, folders = {}, {}
for sp in ["train", "val", "test"]:
    folders[sp] = datasets.ImageFolder(DATASET_DIR / sp, transform=norm_tf)
    loaders[sp] = DataLoader(folders[sp], batch_size=64, shuffle=(sp == "train"), num_workers=2)
assert folders["train"].class_to_idx == folders["val"].class_to_idx == folders["test"].class_to_idx

xb, yb = next(iter(loaders["train"]))
print("Batch tensor:", tuple(xb.shape), "| labels:", tuple(yb.shape), "| classes:", len(folders["train"].classes))
for sp in folders: print(f"{sp:5s}: {len(folders[sp])} images")

per_split = manifest.pivot_table(index="label", columns="split", values="path", aggfunc="count").fillna(0)[["train", "val", "test"]]
display(per_split.describe().round(1))
per_split.plot(kind="bar", stacked=True, figsize=(18, 4), width=0.9, title="Images per class in each split")
plt.xticks(fontsize=6); save_fig("12_split_distribution")

## Save outputs to Drive
Zips the final dataset, so Milestone 2 can start without re-running detection.

In [ ]:
shutil.make_archive(str(PERSIST_DIR / "nutriscan_classification_dataset"), "zip", DATASET_DIR)
shutil.make_archive(str(PERSIST_DIR / "nutriscan_figures"), "zip", FIG_DIR)
save_results()
print("Saved to", PERSIST_DIR); print(sorted(os.listdir(PERSIST_DIR)))

# 6. Actionable Insights

## 6.1 Class imbalance and how to handle it
Class weights are computed so they can be used directly in Milestone 2 if imbalance exists.

In [ ]:
cnt = manifest[manifest.split == "train"].label.value_counts()
ratio_train = cnt.max() / cnt.min()
weights = compute_class_weight("balanced", classes=np.array(class_names), y=manifest[manifest.split == "train"].label.values)
class_weights = dict(zip(class_names, weights.round(4)))
json.dump(class_weights, open(PERSIST_DIR / "class_weights.json", "w"))
print(f"Train imbalance ratio: {ratio_train:.2f} | weight range: {weights.min():.2f} - {weights.max():.2f}")
if ratio_train < 1.5: print("Classes are close to balanced: plain accuracy is acceptable; still report macro-F1.")
else: print("Noticeable imbalance: use class weights / oversampling and judge by macro-F1.")
rset("train_imbalance_ratio", ratio_train)

| Situation | Handling |
|---|---|
| Balanced classes (expected for Food-101) | No resampling needed; report macro-F1 alongside accuracy |
| Moderate imbalance | Class weights (saved in `class_weights.json`), weighted sampler |
| Severe imbalance | Oversample rare classes with augmentation, focal loss |
| Label noise (Food-101 training images are intentionally not cleaned) | Label smoothing, review the highest-loss images after the first model |

## 6.2 Detection accuracy: findings, failures and fixes
Fill in with your actual numbers (they are printed above and stored in `R`).

In [ ]:
print(f"Detection rate: {R['detection_rate']:.1%} (target {TARGET_DET_RATE:.0%}) | fallback: {R['fallback_rate']:.1%}")
print(f"Mean confidence: {R['mean_det_score']:.2f} | median box covers {R['median_box_area_frac']:.0%} of the image")
print(f"Multi-region images: {R['pct_multi_region']:.1f}%")
if "manual_precision_good" in R: print(f"Manual review: {R['manual_precision_good']:.0%} good boxes")

| Where it struggles | Why | How to overcome |
|---|---|---|
| Multiple dishes in one image | Single best box is kept | Keep all boxes above threshold; classify each crop |
| Soups, stews, saucy foods | No clear edges; box may cover bowl or table | Segmentation (FoodSeg103) or bowl-aware prompts |
| Dark / blurry photos | Low scores | Brightness augmentation, lower threshold, test-time augmentation |
| Box covers plate/table | Generic prompts | Tune prompts ("food on a plate"), raise the area filter, upgrade to OWLv2 |
| Close-ups where food fills the frame | Box is almost the whole image | Full-image fallback is acceptable |
| No ground-truth boxes | Cannot compute mAP | Manual review (done); fine-tune YOLO on UEC-Food256 for measurable mAP |

**Other insights (write after running):**
- Image quality findings (dark/blurry share) and their expected impact.
- Visually similar classes likely to be confused in Milestone 2.
- Whether the detector adds value over the plain image for this dataset (many Food-101 photos are already food-centred).

# 7. Plan of Action for the Classification Phase (Milestone 2)

| Step | Plan |
|---|---|
| Input | `classification_dataset/` (train/val/test, 224x224, ImageNet normalisation) |
| Baseline | Small CNN from scratch, to give a reference accuracy |
| Main models | Transfer learning: EfficientNet-B0, ResNet50, MobileNetV2 (mobile deployment) |
| Training | Freeze the backbone for 2-3 epochs, then unfreeze with a lower learning rate; AdamW, cosine schedule, mixed precision, early stopping |
| Augmentation | Random resized crop, flip, colour jitter, blur, light rotation (real user photos vary) |
| Imbalance / noise | Class weights if needed, label smoothing |
| Metrics | Top-1 / Top-5 accuracy, macro-F1, per-class F1, confusion matrix |
| Error analysis | Most-confused pairs, worst classes, failure by lighting/occlusion |
| Selection | Best validation macro-F1, with a latency/size check for mobile; test set used only once |
| Later milestones | Map labels to USDA nutrition data, portion estimation, recommendations, health-app integration |

**Risks:** fine-grained confusion (e.g. similar pasta/cake classes), cuisine coverage limited to 101 classes, portion size and hidden ingredients (oil, sugar) are not visible in photos.

# Conclusion

## Summary of Insights (complete after running)
- **Data:** X images, 101 classes, Y% duplicates/corrupt removed; classes are balanced / imbalanced (ratio Z).
- **Quality:** A% dark, B% blurry; mean RGB close to ImageNet.
- **Detection:** zero-shot OWL-ViT achieved a detection rate of D%, with F% fallbacks; manual review found M% good boxes.
- **Dataset built:** N crops (224x224) split 70/15/15 with no leakage, ready for classification.
- **Next:** transfer learning with EfficientNet/ResNet, evaluated by macro-F1 and confusion analysis.

## Figures saved for the PDF report

| File | Use in report |
|---|---|
| 01_class_distribution | Data overview |
| 02_sample_grid, 03_class_mosaic | EDA: visual inspection |
| 04_image_sizes, 05_brightness_contrast, 05b_quality_examples | EDA: quality |
| 06_detection_samples, 07_threshold_sweep, 08_detection_summary | Detection approach and results |
| 09_manual_qa_grid, 10_failure_cases | Insights and risks |
| 11_before_after | Preprocessing |
| 12_split_distribution | Final dataset |

# Export to HTML for Submission
Run the last cell once the whole notebook has been executed from top to bottom with no errors. If it fails, use **File > Download .ipynb** and convert with the free online ipynb-to-HTML tool.

In [ ]:
# Set NOTEBOOK_PATH to where this notebook is saved in Drive (File > Save a copy in Drive first)
NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/NutriScan_Milestone1.ipynb"
if os.path.exists(NOTEBOOK_PATH):
    !jupyter nbconvert --to html "{NOTEBOOK_PATH}"
else:
    print("Notebook not found at NOTEBOOK_PATH. Adjust it or download the .ipynb and convert it online.")